<a href="https://colab.research.google.com/github/intercambioca-creator/curso-aleman-ia-demo/blob/main/Nervio_optico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install torch torchvision matplotlib opencv-python scikit-learn pillow


In [2]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import os
from PIL import Image
import numpy as np

# Create dummy data directory and files if they don't exist
if not os.path.exists("data/sano"):
    os.makedirs("data/sano")
    # Create a dummy image file for 'sano' class
    dummy_image = Image.fromarray(np.zeros((128, 128, 3), dtype=np.uint8))
    dummy_image.save("data/sano/ejemplo.png")

if not os.path.exists("data/glaucoma"):
    os.makedirs("data/glaucoma")
    # Create a dummy image file for 'glaucoma' class
    dummy_image = Image.fromarray(np.zeros((128, 128, 3), dtype=np.uint8))
    dummy_image.save("data/glaucoma/dummy_glaucoma.png")


transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor()
])

dataset = datasets.ImageFolder(root="data/", transform=transform)
train_loader = DataLoader(dataset, batch_size=16, shuffle=True)


In [3]:
import torch.nn as nn
import torch.nn.functional as F

class OpticDiscNet(nn.Module):
    def __init__(self):
        super(OpticDiscNet, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, 1)
        self.conv2 = nn.Conv2d(16, 32, 3, 1)
        # Corrected input size for fc1 based on calculations (32*62*62)
        self.fc1 = nn.Linear(32*62*62, 64)
        self.fc2 = nn.Linear(64, 2)  # sano vs glaucoma

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)
        # Corrected view operation based on calculations (32*62*62)
        x = x.view(-1, 32*62*62)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

In [4]:
import torch.optim as optim

model = OpticDiscNet()
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

for epoch in range(5):
    for images, labels in train_loader:
        optimizer.zero_grad()
        output = model(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 1, Loss: 0.6971
Epoch 2, Loss: 1.8182
Epoch 3, Loss: 0.7945
Epoch 4, Loss: 0.6979
Epoch 5, Loss: 0.6982


In [6]:
import matplotlib.pyplot as plt
import cv2
import numpy as np
from ipywidgets import interact, IntSlider

# Cargar una imagen de ejemplo
img = cv2.imread("disco optico sano.jpg")
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def ajustar_rgb(r=100, g=100, b=100):
    mod_img = img.copy().astype(np.float32)
    mod_img[:,:,0] = np.clip(mod_img[:,:,0] * (r/100), 0, 255)
    mod_img[:,:,1] = np.clip(mod_img[:,:,1] * (g/100), 0, 255)
    mod_img[:,:,2] = np.clip(mod_img[:,:,2] * (b/100), 0, 255)
    mod_img = mod_img.astype(np.uint8)

    plt.figure(figsize=(12,5))
    plt.subplot(1,2,1)
    plt.imshow(mod_img)
    plt.axis("off")
    plt.title("Imagen modificada")

    plt.subplot(1,2,2)
    for i, color in enumerate(["r","g","b"]):
        plt.hist(mod_img[:,:,i].ravel(), bins=256, color=color, alpha=0.5)
    plt.title("Histograma RGB")
    plt.show()

interact(ajustar_rgb,
         r=IntSlider(min=50,max=150,step=10,value=100),
         g=IntSlider(min=50,max=150,step=10,value=100),
         b=IntSlider(min=50,max=150,step=10,value=100))

interactive(children=(IntSlider(value=100, description='r', max=150, min=50, step=10), IntSlider(value=100, de…

<function __main__.ajustar_rgb(r=100, g=100, b=100)>